# Supplementary classification table: Figures 7B, 7C and 8C

In [ ]:
import pandas as pd

from manuscript_analysis.figures.fig_utils import MATCHED_ABLATION_EXPORTS, RECORDING_SCORE_EXPORTS, get_out_path, report_saved
from manuscript_analysis.figures.score_hist import THRESHOLD

test = pd.read_csv(MATCHED_ABLATION_EXPORTS / 'session_predictions_test.csv').query("feature_set == 'all'")
scores = pd.concat({
    'regular test': test.assign(condition=test['class_name']),
    'insertion': pd.read_csv(RECORDING_SCORE_EXPORTS / 'insertion_model_recording_scores.csv').rename(columns={'class_name': 'condition'}),
    'drug': pd.read_csv(RECORDING_SCORE_EXPORTS / 'drug_test_model_recording_scores.csv'),
}, names=['analysis']).reset_index('analysis')
scores['predicted_ad'] = scores['probability_ad'] >= THRESHOLD

def summarize(keys):
    return scores.groupby(keys)['predicted_ad'].agg(n_scores='size', n_categorized_ad='sum', pct_categorized_ad='mean').reset_index()

stats = pd.concat([summarize('analysis').assign(condition='all'), summarize(['analysis', 'condition'])], ignore_index=True)
stats = stats.assign(n_categorized_ctrl=stats['n_scores'] - stats['n_categorized_ad'], pct_categorized_ctrl=1 - stats['pct_categorized_ad'])

csv_path = get_out_path('sup_7B_7C_8C') / 'classification_stats.csv'
stats.to_csv(csv_path, index=False)
report_saved([csv_path])
stats.style.format({'pct_categorized_ad': '{:.1%}', 'pct_categorized_ctrl': '{:.1%}'})